# Audit fuel-first — 12 septembre 2026

## tl;dr
Les sorties sont du développement hors ligne. 223 observations conservées; MAE temporelle macro restreinte 0,07253 s (linéaire), 0,07031 s (quadratique). A atteint 0,20 L dans son observation mais B est moins coûteux.

## Context & Methods
Contrôles compagnons des scripts, pas nouveau modèle.

### Key Assumptions
Références push antérieures; LOCO par circuit; tests Sebring5/6 inchangés. Sommes de zones, pas autonomie de course. Notebook sans dépendances Jupyter installées : cellules exécutées séquentiellement avec Python; exécution noyau Jupyter non vérifiée.

## Data


In [ ]:
from pathlib import Path
import hashlib
import json
import duckdb
import polars as pl
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/licor").is_dir())
base = root / "data/processed/experimental"
paths = [base / p for p in ("four_circuit_harmonized_v1", "four_circuit_response_candidates_v2", "sebring_fuel_first_decisions_v2_final")]
for folder in paths:
    manifest = json.loads((folder / "manifest.json").read_text())
    for entry in manifest["sources"] + manifest["artifacts"]:
        with Path(entry["path"]).open("rb") as handle:
            assert hashlib.file_digest(handle, "sha256").hexdigest() == entry["sha256"], entry["path"]
print("Source and artifact hashes verified for three final outputs")
original = pl.read_csv(base / "four_circuit_low_data_v1/canonical_response_rows.csv")
harmonized = pl.read_csv(paths[0] / "canonical_response_rows.csv", infer_schema_length=None)
keys = ["observation_id", "circuit_id", "run_id", "lap_number", "zone_id", "action", "planned_action", "fuel_saved_l", "time_lost_s", "quality_tier"]
assert original.select(keys).sort("observation_id").equals(harmonized.select(keys).sort("observation_id"))
print("223 identities, actions, outcomes and quality labels preserved")


## Results
Recalcul indépendant depuis les prédictions (mêmes unités et grain).

In [ ]:
con = duckdb.connect()
query = """WITH folds AS (
 SELECT held_out_circuit, target, model, count(*) AS n,
 avg(abs(predicted-actual)) AS mae
 FROM read_csv_auto(?) GROUP BY ALL
) SELECT target, model, avg(mae) AS macro_mae, sum(n) AS n FROM folds GROUP BY ALL ORDER BY target,model"""
result = con.execute(query, [str(paths[1] / "strict_predictions.csv")]).pl()
stored = pl.read_csv(paths[1] / "macro_metrics.csv").filter(pl.col("cohort") == "strict")
for row in result.iter_rows(named=True):
    prior = stored.filter((pl.col("target") == row["target"]) & (pl.col("model") == row["model"]))
    assert abs(prior["macro_mae"][0] - row["macro_mae"]) < 1e-12
print(result)
decisions = pl.read_csv(paths[2] / "decisions.csv")
assert decisions.filter((pl.col("status") == "target_met") & (~pl.col("actual_reference_feasible"))).height == decisions.filter(pl.col("false_feasible")).height
chosen = decisions.filter((pl.col("menu") == "whole_AB_or_reference_push") & (pl.col("target_fuel_l") == 0.2))
print(chosen.select("model", "reference_fuel_saved_l", "reference_time_lost_s", "time_regret_s"))


## Takeaways
L'arithmétique du budget carburant est testée, mais la marge prudente et la connexion HUD ne sont pas calibrées/intégrées. Les choix mixtes sont hypothétiques. Après installation explicite des outils Jupyter, vérifier le noyau avec `python -m jupyter nbconvert --execute --to notebook --inplace notebooks/fuel_first_development_audit.ipynb`.
